# Google Play Store — Analisi completa (dataset)
Questo notebook esegue un'analisi completa del dataset `googleplaystore.csv`. Produce grafici, tabelle, CSV e un report PDF finale con i principali insight.
**Output**: cartella `googleplay_analysis_outputs/`, file `googleplay_report_full.pdf`.
**Nota**: assicurati che `googleplaystore.csv` sia nella stessa directory del notebook prima di eseguire le celle.


In [ ]:
# SEZIONE 1 - Import e configurazione
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import cm

sns.set(style='whitegrid')

DATA_PATH = 'googleplaystore.csv'
OUT_DIR = '/mnt/data/googleplay_analysis_outputs'
PDF_PATH = '/mnt/data/googleplay_report_full.pdf'
os.makedirs(OUT_DIR, exist_ok=True)

print('Working directory:', os.getcwd())
print('Output directory:', OUT_DIR)


In [ ]:
# SEZIONE 2 - Caricamento dati
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
print('Rows read:', len(df))
# preview
df.head()


In [ ]:
# SEZIONE 3 - Pulizia dati
# Rimuovo duplicati App+Category (se presenti)
if {'App','Category'}.issubset(df.columns):
    before = len(df)
    df = df.drop_duplicates(subset=['App','Category'], keep='first')
    print(f'Removed duplicates: {before - len(df)}')

# Funzioni di parsing
def parse_installs(x):
    try:
        if pd.isna(x): return np.nan
        s = str(x).strip().replace(',','').replace('+','')
        if s.lower() in ['free','varies with device','nan','']: return np.nan
        # some values include non-digit chars; extract digits
        s_digits = ''.join(ch for ch in s if ch.isdigit())
        return int(s_digits) if s_digits!='' else np.nan
    except:
        return np.nan

def parse_price(x):
    try:
        if pd.isna(x): return 0.0
        s = str(x).strip()
        # handle cases like 'Everyone' that might have appeared due to parsing issues
        if s.lower().startswith('everyone'): return 0.0
        s = s.replace('$','').replace('USD','').strip()
        s_digits = ''.join(ch for ch in s if (ch.isdigit() or ch=='.'))
        return float(s_digits) if s_digits!='' else 0.0
    except:
        return 0.0

# Apply parsing
if 'Rating' in df.columns:
    df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
else:
    df['Rating'] = np.nan

if 'Reviews' in df.columns:
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce').fillna(0).astype(int)
else:
    df['Reviews'] = 0

if 'Installs' in df.columns:
    df['Installs_raw'] = df['Installs']
    df['Installs'] = df['Installs_raw'].apply(parse_installs)
else:
    df['Installs'] = np.nan

if 'Price' in df.columns:
    df['Price_raw'] = df['Price']
    df['Price'] = df['Price_raw'].apply(parse_price)
else:
    df['Price'] = 0.0

if 'Type' in df.columns:
    df['Type'] = df['Type'].fillna('Free')
else:
    df['Type'] = df['Price'].apply(lambda p: 'Paid' if p>0 else 'Free')

if 'Category' in df.columns:
    df['Category'] = df['Category'].astype(str).str.strip()
else:
    df['Category'] = 'Unknown'

# Derived columns
df['ln_Installs'] = df['Installs'].replace(0, np.nan).apply(lambda x: np.log1p(x) if pd.notna(x) else np.nan)
df['ln_Reviews'] = df['Reviews'].replace(0, np.nan).apply(lambda x: np.log1p(x) if pd.notna(x) else np.nan)

print('Cleaned dataframe shape:', df.shape)
df[['App','Category','Rating','Reviews','Installs','Price','Type']].head(10)


In [ ]:
# SEZIONE 4 - Statistiche descrittive e aggregazioni per categoria
print('Numeric summary:')
display(df[['Rating','Reviews','Installs','Price']].describe().transpose())

cat_stats = df.groupby('Category').agg(
    AppCount=('App','count') if 'App' in df.columns else ('Category','size'),
    MedianRating=('Rating','median'),
    MeanInstalls=('Installs','mean'),
    MedianInstalls=('Installs','median'),
    TotalInstalls=('Installs','sum'),
    MeanPrice=('Price','mean'),
    FreeShare=('Type', lambda s: (s=='Free').mean())
).reset_index().sort_values('AppCount', ascending=False)

display(cat_stats.head(30))
cat_stats.to_csv(os.path.join(OUT_DIR,'category_stats.csv'), index=False)
print('Saved category_stats.csv')


In [ ]:
# SEZIONE 5 - Visualizzazioni e salvataggio immagini
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Rating distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Rating'].dropna(), bins=30, kde=True)
plt.title('Distribuzione Rating')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'rating_distribution.png'))
plt.close()

# Reviews distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Reviews'].replace(0,np.nan).dropna(), bins=50)
plt.title('Distribuzione Reviews (raw)')
plt.xlabel('Reviews')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'reviews_distribution.png'))
plt.close()

# Installs log distribution
plt.figure(figsize=(8,5))
sns.histplot(df['ln_Installs'].dropna(), bins=40)
plt.title('Distribuzione log(1+Installs)')
plt.xlabel('log(1+Installs)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'installs_log_distribution.png'))
plt.close()

# Scatter rating vs ln_installs
plt.figure(figsize=(8,6))
mask = df['ln_Installs'].notna() & df['Rating'].notna()
plt.scatter(df.loc[mask,'ln_Installs'], df.loc[mask,'Rating'], alpha=0.4, s=10)
plt.title('Rating vs log(1+Installs)')
plt.xlabel('log(1+Installs)')
plt.ylabel('Rating')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'rating_vs_lninstalls.png'))
plt.close()

# Spearman correlation heatmap
numeric = df[['Rating','Reviews','Installs','Price']].copy()
corr = numeric.corr(method='spearman')
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Spearman Correlation')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'spearman_corr.png'))
plt.close()

print('Saved images to', OUT_DIR)


In [ ]:
# SEZIONE 6 - Opportunity score e visualizzazioni per categoria
cat_stats = pd.read_csv(os.path.join(OUT_DIR,'category_stats.csv'))

# compute ranks and opportunity score
cat_stats['rank_appcount'] = cat_stats['AppCount'].rank(ascending=False, method='min')
cat_stats['rank_medianinstalls'] = cat_stats['MedianInstalls'].rank(ascending=False, method='min')
cat_stats['opportunity_score'] = cat_stats['rank_medianinstalls'] - cat_stats['rank_appcount']

cat_stats = cat_stats.sort_values('opportunity_score', ascending=False).reset_index(drop=True)
cat_stats.to_csv(os.path.join(OUT_DIR,'category_stats_with_opportunity.csv'), index=False)

# Save top lists
cat_stats.head(20).to_csv(os.path.join(OUT_DIR,'category_stats_top20.csv'), index=False)

# Plot top competition and interest
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10,6))
top_comp = cat_stats.sort_values('AppCount', ascending=False).head(12)
sns.barplot(data=top_comp, x='AppCount', y='Category', palette='viridis')
plt.title('Top categorie per numero di app (competizione)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'top_competition.png'))
plt.close()

plt.figure(figsize=(10,6))
top_interest = cat_stats.sort_values('MedianInstalls', ascending=False).head(12)
sns.barplot(data=top_interest, x='MedianInstalls', y='Category', palette='crest')
plt.title('Top categorie per interesse (MedianInstalls)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'top_interest.png'))
plt.close()

print('Opportunity-based category stats saved and plots generated.')
display(cat_stats.head(10))


In [ ]:
# SEZIONE 7 - Top apps per top-3 categorie (by opportunity)
top5 = cat_stats.head(5)
top3 = top5['Category'].head(3).tolist()
top3_details = {}
for cat in top3:
    apps = df[df['Category']==cat].sort_values('Installs', ascending=False).head(20)
    cols = ['App','Category','Installs','Rating','Reviews','Type','Price']
    apps_out = apps[cols].copy()
    apps_out['Installs'] = apps_out['Installs'].fillna(0).astype(int)
    csvname = os.path.join(OUT_DIR, f'top_apps_{cat.replace("/","_")}.csv')
    apps_out.to_csv(csvname, index=False)
    top3_details[cat] = csvname
    print('\nTop apps for category:', cat)
    display(apps_out.head(10))

print('\nTop3 categories:', top3)


In [ ]:
# SEZIONE 8 - Test statistici (Paid vs Free)
from scipy import stats
paid = df[df['Type']=='Paid']
free = df[df['Type']=='Free']

r_paid = paid['Rating'].dropna()
r_free = free['Rating'].dropna()

try:
    stat_rating, p_rating = stats.mannwhitneyu(r_paid, r_free, alternative='two-sided')
except Exception as e:
    stat_rating, p_rating = (np.nan, np.nan)

ip_paid = paid['ln_Installs'].dropna()
ip_free = free['ln_Installs'].dropna()
try:
    stat_installs, p_installs = stats.mannwhitneyu(ip_paid, ip_free, alternative='two-sided')
except Exception as e:
    stat_installs, p_installs = (np.nan, np.nan)

tests_summary = pd.DataFrame([
    {'Metric':'Rating','stat':stat_rating,'p_value':p_rating,'n_paid':len(r_paid),'n_free':len(r_free)},
    {'Metric':'ln_Installs','stat':stat_installs,'p_value':p_installs,'n_paid':len(ip_paid),'n_free':len(ip_free)}
])
display(tests_summary)


In [ ]:
# SEZIONE 9 - Forecast monetizzazione (12 mesi) - top3 categories
scenarios = {
    'conservative': {'market_share':0.0005, 'conv_rate':0.01, 'arpu_m':1.0, 'ad_m':0.10},
    'base': {'market_share':0.005, 'conv_rate':0.03, 'arru_m':2.5, 'ad_m':0.25},  # note: 'arru_m' typo avoided below
    'aggressive': {'market_share':0.02, 'conv_rate':0.05, 'arpu_m':5.0, 'ad_m':0.50},
}

# Fix for base scenario dict key naming (ensure 'arpu_m' exists)
scenarios['base']['arpu_m'] = scenarios['base'].pop('arru_m') if 'arru_m' in scenarios['base'] else 2.5

forecast_rows = []
for cat in top3:
    total_installs = int(cat_stats.loc[cat_stats['Category']==cat,'TotalInstalls'].values[0] if len(cat_stats.loc[cat_stats['Category']==cat,'TotalInstalls'].values)>0 else 0)
    for scen_name, params in scenarios.items():
        ms = params['market_share']; conv = params['conv_rate']; arpu = params['arpu_m']; adm = params['ad_m']
        captured_installs = int(total_installs * ms)
        mau = captured_installs
        paid_users = int(mau * conv)
        monthly_sub_rev = paid_users * arpu
        annual_sub_rev = monthly_sub_rev * 12
        monthly_ad_rev = mau * adm
        annual_ad_rev = monthly_ad_rev * 12
        total_annual_rev = annual_sub_rev + annual_ad_rev
        forecast_rows.append({
            'Category':cat,
            'Scenario':scen_name,
            'Category_TotalInstalls': total_installs,
            'Captured_Installs_Y1': captured_installs,
            'MAU_est': mau,
            'PaidUsers_est': paid_users,
            'Monthly_Sub_Rev_USD': round(monthly_sub_rev,2),
            'Annual_Sub_Rev_USD': round(annual_sub_rev,2),
            'Monthly_Ad_Rev_USD': round(monthly_ad_rev,2),
            'Annual_Ad_Rev_USD': round(annual_ad_rev,2),
            'Total_Annual_Rev_USD': round(total_annual_rev,2),
            'Assumptions': f"ms={ms}, conv={conv}, arpu_m={arpu}, ad_m={adm}"
        })

forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv(os.path.join(OUT_DIR,'forecast_top3_categories.csv'), index=False)
display(forecast_df)


In [ ]:
# SEZIONE 10 - Riepilogo file creati
import glob
files = glob.glob(os.path.join(OUT_DIR,'*'))
print('Files in output dir:')
for f in files:
    print('-', f)
print('\nNotebook execution complete.')


In [ ]:
# SEZIONE 11 - Generazione report PDF (inclusi grafici e tabelle)
styles = getSampleStyleSheet()
doc = SimpleDocTemplate(PDF_PATH, pagesize=A4)
story = []

story.append(Paragraph("Google Play Store - Analisi Strategica (Completo)", styles['Title']))
story.append(Spacer(1,12))
story.append(Paragraph(f"Totale app analizzate: {len(df)}  |  Categorie uniche: {df['Category'].nunique()}", styles['Normal']))
story.append(Spacer(1,12))

# Add images (if present)
image_list = [
    ('Distribuzione Rating', os.path.join(OUT_DIR,'rating_distribution.png')),
    ('Distribuzione Reviews', os.path.join(OUT_DIR,'reviews_distribution.png')),
    ('Distribuzione log(Installs)', os.path.join(OUT_DIR,'installs_log_distribution.png')),
    ('Rating vs log(Installs)', os.path.join(OUT_DIR,'rating_vs_lninstalls.png')),
    ('Spearman Correlation', os.path.join(OUT_DIR,'spearman_corr.png')),
    ('Top Competition', os.path.join(OUT_DIR,'top_competition.png')),
    ('Top Interest', os.path.join(OUT_DIR,'top_interest.png')),
    ('Opportunity Map', os.path.join(OUT_DIR,'opportunity_map.png')) # may not exist; safe to ignore if missing
]

for title, imgpath in image_list:
    try:
        story.append(Paragraph(title, styles['Heading2']))
        story.append(Image(imgpath, width=16*cm, height=9*cm))
        story.append(Spacer(1,12))
    except Exception as e:
        # Missing image — skip
        pass

# Top 3 categories tables
story.append(PageBreak())
story.append(Paragraph('Profilo competitivo - Top 3 categorie', styles['Heading1']))
for cat in top3:
    try:
        tbl_df = pd.read_csv(os.path.join(OUT_DIR, f'top_apps_{cat.replace("/","_")}.csv'))
    except:
        continue
    story.append(Paragraph(cat, styles['Heading2']))
    data = [list(tbl_df.columns)] + tbl_df.values.tolist()
    tbl = Table(data, colWidths=[7*cm, 2.5*cm, 1.5*cm, 2.5*cm, 1.5*cm, 1.5*cm])
    tbl.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#D3D3D3')),
                             ('GRID',(0,0),(-1,-1),0.25,colors.grey),
                             ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold')]))
    story.append(tbl)
    story.append(Spacer(1,12))

# Forecast table
story.append(PageBreak())
story.append(Paragraph('Forecast monetizzazione (12 mesi)', styles['Heading1']))
fcols = ['Category','Scenario','Captured_Installs_Y1','MAU_est','PaidUsers_est','Annual_Sub_Rev_USD','Annual_Ad_Rev_USD','Total_Annual_Rev_USD']
# fallback if columns differ
fcols = [c for c in forecast_df.columns if c in forecast_df.columns]
data = [list(forecast_df.columns)] + forecast_df.values.tolist()
ftbl = Table(data, colWidths=[3*cm]*len(forecast_df.columns))
ftbl.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#D3D3D3')),
                          ('GRID',(0,0),(-1,-1),0.25,colors.grey),
                          ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold')]))
story.append(ftbl)
story.append(Spacer(1,12))

# Conclusions
conclusions = """Conclusioni strategiche:
- BUSINESS: concorrenza moderata e mercato ampio → ottimo candidato per MVP B2B sostenibile.
- MEDICAL: nicchia con alte barriere (fiducia) ma opportunità per app green-health.
- PERSONALIZATION: grande volume ma forte competizione; differenziazione tramite design/temi eco.
Raccomandazione: partire con MVP in BUSINESS o MEDICAL; modello freemium.
"""
story.append(Paragraph(conclusions.replace('\n','<br/>'), styles['Normal']))

doc.build(story)
print('PDF generato:', PDF_PATH)
